### Import ans setUp

In [16]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['GROQ_API_KEY']:
    print("Api key is set")

Api key is set


In [17]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0
)

### **RAG Implementation with Your Own Text Data**

#### **Step 1 : Preparing Documents for your text**

In [18]:
from langchain_core.documents import Document

# your text data
my_text : str = """Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]
High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.
The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers have used techniques including state space search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind and Meta, aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human.[3
Artificial intelligence was founded as an academic discipline in 1956,[4] and the field went through multiple cycles of optimism throughout its history,[5][6] followed by periods of disappointment and loss of funding, known as AI winters.[7][8] Funding and interest increased substantially after 2012, when graphics processing units began being used to accelerate neural networks, and deep learning outperformed previous AI techniques.[9] This growth accelerated further after 2017 with the transformer architecture.[10] In the 2020s, an AI boom has coincided with advances in generative AI, which allowed for the creation and modification of media. In addition to AI safety and unintended consequences and harms from the use of AI, ethical concerns, AI's long-term effects, and potential existential risks have prompted discussions of AI regulation."""

# convert text to documents (list)
docs = [Document(page_content=my_text, metadata={"source" :"dhanushkumar", "documentId" : "Docs1"})]

#### **Step 2 : Splitting the documents into Chuncks**

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100, # chunk size
    chunk_overlap = 50 # take 50 character from previous chunk so data flow cant  be break
)

chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'dhanushkumar', 'documentId': 'Docs1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically'),
 Document(metadata={'source': 'dhanushkumar', 'documentId': 'Docs1'}, page_content='computational systems to perform tasks typically associated with human intelligence, such as'),
 Document(metadata={'source': 'dhanushkumar', 'documentId': 'Docs1'}, page_content='associated with human intelligence, such as learning, reasoning, problem-solving, perception, and'),
 Document(metadata={'source': 'dhanushkumar', 'documentId': 'Docs1'}, page_content='reasoning, problem-solving, perception, and decision-making. It is a field of research in'),
 Document(metadata={'source': 'dhanushkumar', 'documentId': 'Docs1'}, page_content='and decision-making. It is a field of research in engineering, mathematics and computer science'),
 Document(metadata={'source': 'dhanushkumar', 'documentId': 'Docs1'}, page_content='in en

#### **Step 3 : Creating the embeddings from chunks**

In [20]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### **Step 4: Create and Store a Embeddings in Vector Store**

In [26]:
from langchain_community.vectorstores import Chroma

vectoreStore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

### **Similarity Search**

In [27]:
vectoreStore.similarity_search("what is ai?", k=3)

[Document(metadata={'source': 'dhanushkumar', 'documentId': 'Docs1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that'),
 Document(metadata={'documentId': 'Docs1', 'source': 'dhanushkumar'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically'),
 Document(metadata={'documentId': 'Docs1', 'source': 'dhanushkumar'}, page_content='other fields.[2] Some companies, such as OpenAI, Google DeepMind and Meta, aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human.[3')]

### **Talk To LLM**

In [29]:
context = vectoreStore.similarity_search("what is ai?", k=3)

response = model.invoke(f"what is ai?, you can aswer using following conetx :{context}")
print(response.content)

Based on the provided context, here's an explanation of AI:

**Artificial Intelligence (AI)** is the capability of **computational systems** to perform tasks typically associated with **human intelligence**, such as:

1. **Learning**: Acquiring new knowledge or skills.
2. **Reasoning**: Drawing conclusions or making decisions based on available information.
3. **Problem-solving**: Identifying and resolving complex issues.
4. **Perception**: Interpreting and understanding sensory data (e.g., images, sounds).
5. **Decision-making**: Choosing the best course of action based on available information.

AI is a field of research that combines **engineering**, **mathematics**, and **computer science** to create intelligent systems. Some companies, like **OpenAI**, **Google DeepMind**, and **Meta**, are working towards developing **Artificial General Intelligence (AGI)**, which is AI that can perform virtually any cognitive task at least as well as a human.
